# LSTM AQI FORECAST TRAINING

Trains a Deep Learning LSTM model for 24-hour AQI prediction using hybrid datasets.

In [11]:
import pandas as pd
import numpy as np
import os
import pickle

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

## Configuration

In [12]:
DATA_PATH = "../data/final_hybrid_india_aqi_dataset (3).csv"
MODEL_SAVE_PATH = "../models/lstm_model.h5"
SCALER_SAVE_PATH = "../models/scaler.pkl"

WINDOW_SIZE = 30
EPOCHS = 25
BATCH_SIZE = 32

## Load Data

In [13]:
print("Loading dataset...")
df = pd.read_csv(DATA_PATH)

print("Initial Shape:", df.shape)

df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(['City', 'Date']).reset_index(drop=True)

Loading dataset...
Initial Shape: (24146, 14)


## Handle Missing Values

In [14]:
print("\nHandling missing values...")

# Defining columns to fill (PM2.5, AQI, future_aqi, etc.)
feature_cols = ['PM2.5', 'PM10', 'NO2', 'CO', 'SO2', 'O3', 'wind_speed', 'temperature', 'humidity', 'violations_7d', 'AQI']
target_col = 'future_aqi'
cols_to_fill = feature_cols + [target_col]

# We use transform to fill NAs per group while preserving all columns (including 'City')
df[cols_to_fill] = df.groupby('City')[cols_to_fill].transform(lambda x: x.ffill().bfill())
df = df.dropna()

print("Remaining nulls:")
print(df.isnull().sum())


Handling missing values...
Remaining nulls:
City             0
Date             0
PM2.5            0
PM10             0
NO2              0
CO               0
SO2              0
O3               0
wind_speed       0
violations_7d    0
temperature      0
humidity         0
AQI              0
future_aqi       0
dtype: int64


## Feature Selection

In [15]:
feature_cols = [
    'PM2.5',
    'PM10',
    'NO2',
    'CO',
    'SO2',
    'O3',
    'wind_speed',
    'temperature',
    'humidity',
    'violations_7d',
    'AQI'
]

target_col = 'future_aqi'

# Ensure City and Date are part of the subset
df = df[['City', 'Date'] + feature_cols + [target_col]]
print("Final Columns:", df.columns)
print("Dataframe Head:\n", df.head())

Final Columns: Index(['City', 'Date', 'PM2.5', 'PM10', 'NO2', 'CO', 'SO2', 'O3', 'wind_speed',
       'temperature', 'humidity', 'violations_7d', 'AQI', 'future_aqi'],
      dtype='str')
Dataframe Head:
         City       Date   PM2.5    PM10    NO2     CO    SO2      O3  \
0  Ahmedabad 2015-01-29   83.13  122.41  28.71   6.93  49.52   59.76   
1  Ahmedabad 2015-01-30   79.84  122.41  28.68  13.85  48.49   97.07   
2  Ahmedabad 2015-01-31   94.52  122.41  32.66  24.39  67.39  111.33   
3  Ahmedabad 2015-02-01  135.99  122.41  42.08  43.48  75.23  102.70   
4  Ahmedabad 2015-02-02  178.33  122.41  35.31  54.56  55.04  107.38   

   wind_speed  temperature   humidity  violations_7d    AQI  future_aqi  
0    2.352371    29.085095  67.620599              4  209.0       328.0  
1    1.249389    39.914403  84.626358              6  328.0       514.0  
2    0.948151    23.066854  33.254225             10  514.0       782.0  
3    0.308424    25.310558  77.137775             15  782.0       9

## Scaling

In [16]:
scaler = MinMaxScaler()

df[feature_cols + [target_col]] = scaler.fit_transform(
    df[feature_cols + [target_col]]
)

os.makedirs("../models", exist_ok=True)
with open(SCALER_SAVE_PATH, "wb") as f:
    pickle.dump(scaler, f)

print("Scaler saved.")

Scaler saved.


## Create Sequences

In [17]:
def create_sequences(data, window_size):
    X, y = [], []
    cities = data['City'].unique()

    for city in cities:
        city_df = data[data['City'] == city]
        values = city_df[feature_cols + [target_col]].values

        for i in range(len(values) - window_size):
            X.append(values[i:i+window_size, :-1])
            y.append(values[i+window_size-1, -1])

    return np.array(X), np.array(y)

X, y = create_sequences(df, WINDOW_SIZE)
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (21385, 30, 11)
y shape: (21385,)


## Train-Validation Split

In [18]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    shuffle=False
)

print("Training samples:", X_train.shape[0])
print("Validation samples:", X_val.shape[0])

Training samples: 17108
Validation samples: 4277


## Build and Train LSTM Model

In [19]:
model = Sequential()
model.add(LSTM(64, return_sequences=True, input_shape=(WINDOW_SIZE, len(feature_cols))))
model.add(Dropout(0.2))
model.add(LSTM(32))
model.add(Dropout(0.2))
model.add(Dense(16, activation='relu'))
model.add(Dense(1))

model.compile(optimizer='adam', loss='mse')
model.summary()

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop]
)

C:\Users\LENOVO\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 30, 64)         │        19,456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 32,417 (126.63 KB)

 Trainable params: 32,417 (126.63 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/25
535/535 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step - loss: 0.0022 - val_loss: 5.6260e-04
Epoch 2/25
535/535 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - loss: 0.0017 - val_loss: 5.3321e-04
Epoch 3/25
535/535 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - loss: 0.0016 - val_loss: 4.0869e-04
Epoch 4/25
535/535 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - loss: 0.0014 - val_loss: 3.7936e-04
Epoch 5/25
535/535 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - loss: 0.0013 - val_loss: 3.3494e-04
Epoch 6/25
535/535 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - loss: 0.0013 - val_loss: 3.6098e-04
Epoch 7/25
535/535 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - loss: 0.0012 - val_loss: 3.0984e-04
Epoch 8/25
535/535 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - loss: 0.0012 - val_loss: 3.0104e-04
Epoch 9/25
535/535 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - loss: 0.0012 - val_loss: 3.0652e-04
Epoch 10/25
535/535 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - loss: 0.0012 - val_loss: 3.0218e-04
Epoch 11/25
535/535 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - loss: 0.0011 - val_loss: 3.0187e

## Save Model

In [ ]:
model.save(MODEL_SAVE_PATH)
print("\nModel saved successfully.")
print("Training complete.")